# TFG V6: QFT-based interference for window search

This notebook explores the same contiguous free-window search problem studied in V4 and V5, but V6 is now kept as a pure QFT-based interference experiment. The index register is initialized directly in the valid search domain, so the initial state is the uniform superposition over window start positions \(i=0,\ldots,W-1\), and the grid itself remains classical.

The V6 idea is to encode structural information about each candidate window as a diagonal phase profile on the index register and then apply a QFT readout. There is no Grover diffusion operator and no local mixer in this version.

The current version keeps the V4/V5 case-study workflow, but the optimisation is reduced to \(\theta\) and repetitions \(r\). There is no \(\beta\), because \(\beta\) was the mixer angle and the mixer has been removed.

Implemented phase modes:

- `cost_linear`: V4-style phase proportional to the number of occupied cells in the window, \(C(i)\).
- `cost_exponential`: nonlinear penalty controlled by `exp_base`; it separates higher-cost windows more aggressively.
- `delta_cost`: V5-style signed domain-wall signal \(\Delta C(i)=C(i+1)-C(i)\), useful for detecting descents into freer regions.
- `fourier_encoded`: classical dominant Fourier component of the cost profile compiled as a diagonal phase pattern.

Circuit flow:

1. Prepare \(\frac{1}{\sqrt W}\sum_i |i\rangle\).
2. Apply the diagonal phase oracle \(r\) times.
3. Apply a final QFT or inverse QFT on the valid \(W\)-dimensional subspace.
4. Measure/interprete the index register as the window-start distribution.


In [ ]:
import os
if not os.path.exists("TFG_V6.ipynb"):
    raise RuntimeError("Run this notebook from the folder that contains TFG_V6.ipynb; output folders are intentionally local to that folder.")
os.environ.setdefault("MPLCONFIGDIR", os.path.join(os.getcwd(), ".matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", os.path.join(os.getcwd(), ".cache"))
os.environ.setdefault("MPLBACKEND", "Agg")

from math import prod
from pathlib import Path
import csv
import re
import time

import numpy as np
import matplotlib.pyplot as plt

import qiskit
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import MCXGate, PhaseGate, UnitaryGate

plt.rcParams.update({
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
})

COLOR_VALID = "#2ecc71"
COLOR_INVALID = "#e74c3c"
COLOR_BASELINE = "#7f8c8d"

V6_OUTPUT_DIR = Path("TFG_V6_Analysis")
V6_OUTPUT_DIR.mkdir(exist_ok=True)

print(qiskit.__version__)


In [ ]:
# =========================================================
# Parametros configurables del experimento principal
# =========================================================

N = [8]
M = [2]
occupied_coords = [(0,), (1,), (2,), (6,), (7,)]

# Parametros QFT
phase_mode = "cost_linear"   # "cost_linear", "cost_exponential", "delta_cost", "fourier_encoded"
theta = np.pi / 3
exp_base = 2.0               # solo para cost_exponential
freq_target = None           # None = dominante automatico, int = frecuencia fija

# Parametros de lectura QFT
repetitions = 4
use_qft = True
qft_inverse = False          # True: usar iQFT en lugar de QFT


In [ ]:
# =========================================================
# Geometria ND y utilidades clasicas
# =========================================================

def validate_problem(N, M):
    if len(N) != len(M):
        raise ValueError("N and M must have the same dimension.")
    for d, (n_d, m_d) in enumerate(zip(N, M)):
        if n_d <= 0 or m_d <= 0:
            raise ValueError(f"N[{d}] and M[{d}] must be positive.")
        if m_d > n_d:
            raise ValueError(f"M[{d}] cannot be greater than N[{d}].")


def coord_to_index(coord, dims):
    """Converts ND coordinates to a row-major linear index.
    Example with dims = [4,4]:
    (0,0) -> 0    (0,1) -> 1    (0,2) -> 2    (0,3) -> 3
    (1,0) -> 4    (1,1) -> 5    (1,2) -> 6    (1,3) -> 7
    (2,0) -> 8    (2,1) -> 9    (2,2) -> 10   (2,3) -> 11
    (3,0) -> 12   (3,1) -> 13   (3,2) -> 14   (3,3) -> 15
    """
    idx_lin = 0
    stride = 1
    for d in reversed(range(len(dims))):
        idx_lin += coord[d] * stride
        stride *= dims[d]
    return idx_lin


def index_to_coord(index, dims):
    """Converts a row-major linear index to ND coordinates.
    Example with dims = [4,4]:
    0 -> (0,0)    1 -> (0,1)    2 -> (0,2)    3 -> (0,3)
    4 -> (1,0)    5 -> (1,1)    6 -> (1,2)    7 -> (1,3)
    8 -> (2,0)    9 -> (2,1)    10 -> (2,2)   11 -> (2,3)
    12 -> (3,0)   13 -> (3,1)    14 -> (3,2)   15 -> (3,3)
    """
    coord = [0] * len(dims)
    rem = index
    for d in reversed(range(len(dims))):
        coord[d] = rem % dims[d]
        rem //= dims[d]
    return tuple(coord)


def valid_starts_nd(N, M):
    """Valid starting coordinates for a window M inside N."""
    return list(np.ndindex(tuple(N[d] - M[d] + 1 for d in range(len(N)))))


def window_qubits_nd(start, N, M):
    """Linear indices of the cells covered by the window starting at start."""
    qubits = []
    for offset in np.ndindex(tuple(M)):
        coord = tuple(start[d] + offset[d] for d in range(len(N)))
        qubits.append(coord_to_index(coord, N))
    return qubits


def normalize_coord(coord, D):
    if D == 1 and isinstance(coord, int):
        return (coord,)
    return tuple(coord)


def build_grid_bits(N, occupied_coords):
    """Returns a classical vector with 1 on occupied cells and 0 on free cells."""
    D = len(N)
    grid = [0] * prod(N)
    for raw_coord in occupied_coords:
        coord = normalize_coord(raw_coord, D)
        if len(coord) != D:
            raise ValueError(f"Coordinate {coord} does not have dimension {D}.")
        for d, x in enumerate(coord):
            if x < 0 or x >= N[d]:
                raise ValueError(f"Coordinate {coord} is outside the grid N={N}.")
        grid[coord_to_index(coord, N)] = 1
    return grid


def compute_window_cost_classical(grid_bits, start, N, M):
    """C(i): number of ones in the window associated with start."""
    return sum(grid_bits[q] for q in window_qubits_nd(start, N, M))


def window_string_classical(grid_bits, start, N, M):
    return ''.join(str(grid_bits[q]) for q in window_qubits_nd(start, N, M))


def get_valid_indices(grid_bits, starts, N, M):
    return [i for i, start in enumerate(starts) if compute_window_cost_classical(grid_bits, start, N, M) == 0]


def gray_order_valid(W, IDX):
    """
    000 -> 0
    001 -> 1
    011 -> 3
    010 -> 2
    110 -> 6
    111 -> 7
    101 -> 5
    100 -> 4
    """
    gray_full = [t ^ (t >> 1) for t in range(2**IDX)]
    return [g for g in gray_full if g < W]


def format_nd_array_from_bits(bitstring, dims):
    arr = np.array(list(bitstring), dtype=str).reshape(tuple(dims))
    return np.array2string(arr, separator=' ').replace("'", "")


In [ ]:
# =========================================================
# Bloques cuanticos: QFT y fase diagonal
# =========================================================

def prepare_valid_index_superposition(qc, idx, W):
    """Prepares 1/sqrt(W) sum_{i=0}^{W-1} |i>, even when W is not a power of 2."""
    IDX = len(idx)
    amps = np.zeros(2**IDX, dtype=complex)
    amps[:W] = 1 / np.sqrt(W)
    qc.initialize(amps, idx)


def append_mcx(qc, controls, target):
    """Modern MCX through MCXGate, without deprecated arguments such as mode='noancilla'."""
    if len(controls) == 0:
        qc.x(target)
    elif len(controls) == 1:
        qc.cx(controls[0], target)
    else:
        qc.append(MCXGate(num_ctrl_qubits=len(controls)), controls + [target])


def apply_window_loader(qc, n, idx, m, starts, N, M, order_valid):
    """
    Reversible loader kept for reference with previous versions. The pure V6
    circuit compiles the classical phase profile directly on idx and does not
    use a mixer.
    """
    IDX = len(idx)
    current_zero_mask = [False] * IDX

    for i in order_valid:
        bits = [(i >> b) & 1 for b in range(IDX)]
        target_zero_mask = [bits[b] == 0 for b in range(IDX)]

        for b in range(IDX):
            if current_zero_mask[b] != target_zero_mask[b]:
                qc.x(idx[b])
                current_zero_mask[b] = target_zero_mask[b]

        for j, n_pos in enumerate(window_qubits_nd(starts[i], N, M)):
            controls = [idx[b] for b in range(IDX)] + [n[n_pos]]
            append_mcx(qc, controls, m[j])

    for b in range(IDX):
        if current_zero_mask[b]:
            qc.x(idx[b])


def apply_phase_to_basis_state(qc, idx, basis_state, angle):
    """Applies exp(i angle) to the computational state |basis_state> of the idx register."""
    if abs(angle) < 1e-15:
        return

    IDX = len(idx)
    zero_qubits = [q for bit, q in enumerate(idx) if ((basis_state >> bit) & 1) == 0]
    for q in zero_qubits:
        qc.x(q)

    if IDX == 1:
        qc.p(angle, idx[0])
    else:
        gate = PhaseGate(angle).control(IDX - 1)
        qc.append(gate, list(idx[:-1]) + [idx[-1]])

    for q in reversed(zero_qubits):
        qc.x(q)


def apply_qft(qc, register, inverse=False, do_swaps=True):
    """Standard 2^k-dimensional QFT reference implementation."""
    qubits = list(register)
    n = len(qubits)

    if inverse:
        if do_swaps:
            for j in range(n // 2):
                qc.swap(qubits[j], qubits[n - j - 1])
        for j in reversed(range(n)):
            for m in reversed(range(j + 1, n)):
                qc.cp(-np.pi / (2 ** (m - j)), qubits[m], qubits[j])
            qc.h(qubits[j])
        return

    for j in range(n):
        qc.h(qubits[j])
        for m in range(j + 1, n):
            qc.cp(np.pi / (2 ** (m - j)), qubits[m], qubits[j])
    if do_swaps:
        for j in range(n // 2):
            qc.swap(qubits[j], qubits[n - j - 1])


def apply_qft_on_valid_subspace(qc, idx, W, inverse=False):
    """
    Applies a W-dimensional QFT embedded in the 2^IDX index-register Hilbert space.
    This avoids leakage into padded states when W is not a power of two.
    """
    IDX = len(idx)
    dim = 2**IDX
    if W <= 1:
        qc.append(UnitaryGate(np.eye(dim, dtype=complex), label="iQFT_W" if inverse else "QFT_W"), list(idx))
        return

    j, k = np.meshgrid(np.arange(W), np.arange(W), indexing="ij")
    F = np.exp(2j * np.pi * j * k / W) / np.sqrt(W)
    if inverse:
        F = F.conj().T

    U = np.eye(dim, dtype=complex)
    U[:W, :W] = F
    qc.append(UnitaryGate(U, label="iQFT_W" if inverse else "QFT_W"), list(idx))


def compute_delta_costs_classical(costs, shift=1, boundary_mode="zero"):
    """V5-style signed domain-wall signal between neighbouring window costs."""
    W = len(costs)
    delta_costs = []
    for i in range(W):
        j = i + int(shift)
        if j < W:
            delta_costs.append(float(costs[j] - costs[i]))
        elif boundary_mode == "zero":
            delta_costs.append(float(0 - costs[i]))
        elif boundary_mode == "same":
            delta_costs.append(0.0)
        elif boundary_mode == "wrap":
            delta_costs.append(float(costs[j % W] - costs[i]))
        else:
            raise ValueError(f"Unknown boundary_mode: {boundary_mode}")
    return delta_costs


def compute_phase_profile(costs, theta, exp_base=2.0, phase_mode="cost_linear", freq_target=None,
                          shift=1, boundary_mode="zero"):
    """Calcula phi_i de forma clasica para compilar un oraculo diagonal."""
    costs_arr = np.array(costs, dtype=float)

    if phase_mode == "cost_linear":
        phases = theta * costs_arr
        return phases.tolist(), freq_target

    if phase_mode == "cost_exponential":
        phases = theta * (np.power(exp_base, costs_arr) - 1.0)
        return phases.tolist(), freq_target

    if phase_mode == "delta_cost":
        delta_costs = np.asarray(
            compute_delta_costs_classical(costs_arr, shift=shift, boundary_mode=boundary_mode),
            dtype=float,
        )
        return (theta * delta_costs).tolist(), freq_target

    if phase_mode == "fourier_encoded":
        W = len(costs_arr)
        if W == 1:
            return [0.0], 0
        freqs = np.fft.fft(costs_arr)
        if freq_target is None:
            freqs_no_dc = freqs.copy()
            freqs_no_dc[0] = 0
            freq_target = int(np.argmax(np.abs(freqs_no_dc)))
        freq_target = int(freq_target) % W
        i = np.arange(W)
        carrier = np.exp(2j * np.pi * freq_target * i / W)
        phases = theta * np.real(freqs[freq_target] * carrier)
        return phases.tolist(), freq_target

    raise ValueError("Unknown phase_mode.")


def apply_phase_oracle(qc, idx, phi_profile, W, label="U_phi"):
    """Aplica una fase diagonal exp(i phi_i) sobre cada estado |i>."""
    for i in range(W):
        apply_phase_to_basis_state(qc, idx, i, phi_profile[i])
    qc.barrier(label=label)


In [ ]:
# =========================================================
# Construccion del circuito
# =========================================================

def build_v6_circuit(
    N, M, occupied_coords,
    theta, exp_base, repetitions,
    phase_mode="cost_linear",
    freq_target=None,
    use_qft=True,
    qft_inverse=False,
    add_barriers=True,
    shift=1,
    boundary_mode="zero",
):
    validate_problem(N, M)
    D = len(N)
    starts = valid_starts_nd(N, M)
    W = len(starts)
    IDX = int(np.ceil(np.log2(W))) if W > 1 else 1
    order_valid = gray_order_valid(W, IDX)
    grid_bits = build_grid_bits(N, occupied_coords)
    costs = [compute_window_cost_classical(grid_bits, start, N, M) for start in starts]
    valids = [1 if c == 0 else 0 for c in costs]
    phi_profile, selected_freq = compute_phase_profile(
        costs=costs,
        theta=theta,
        exp_base=exp_base,
        phase_mode=phase_mode,
        freq_target=freq_target,
        shift=shift,
        boundary_mode=boundary_mode,
    )

    idx = QuantumRegister(IDX, "i")
    qc = QuantumCircuit(idx)
    prepare_valid_index_superposition(qc, idx, W)
    if add_barriers:
        qc.barrier(label="init")

    for r in range(repetitions):
        apply_phase_oracle(qc, idx, phi_profile, W, label=f"U_phi[{r}]")
        if add_barriers:
            qc.barrier(label=f"phase layer {r}")

    if use_qft:
        apply_qft_on_valid_subspace(qc, idx, W, inverse=qft_inverse)
        if add_barriers:
            qc.barrier(label="iQFT final" if qft_inverse else "QFT final")

    metadata = {
        "D": D,
        "N": list(N),
        "M": list(M),
        "starts": starts,
        "W": W,
        "IDX": IDX,
        "order_valid": order_valid,
        "grid_bits": grid_bits,
        "occupied_coords": [normalize_coord(c, D) for c in occupied_coords],
        "theta": theta,
        "exp_base": exp_base,
        "repetitions": repetitions,
        "phase_mode": phase_mode,
        "freq_target": selected_freq,
        "use_qft": use_qft,
        "qft_inverse": qft_inverse,
        "shift": shift,
        "boundary_mode": boundary_mode,
        "delta_costs": compute_delta_costs_classical(costs, shift=shift, boundary_mode=boundary_mode),
        "phi_profile": phi_profile,
        "costs": costs,
        "valids": valids,
    }
    return qc, metadata


def index_probabilities_from_statevector(sv, metadata):
    IDX = metadata["IDX"]
    probs = np.zeros(2**IDX, dtype=float)
    idx_mask = (1 << IDX) - 1
    for basis_idx, amp in enumerate(sv.data):
        probs[basis_idx & idx_mask] += float(abs(amp) ** 2)
    return probs


def p_valid_from_circuit(qc, metadata):
    sv = Statevector.from_instruction(qc)
    probs = index_probabilities_from_statevector(sv, metadata)
    valid_indices = [i for i, v in enumerate(metadata["valids"]) if v == 1]
    return float(sum(probs[i] for i in valid_indices)), probs


In [ ]:
# =========================================================
# ANALYSIS 1 — Build and display main circuit
# =========================================================

qc_v6, meta_v6 = build_v6_circuit(
    N=N,
    M=M,
    occupied_coords=occupied_coords,
    theta=theta,
    exp_base=exp_base,
    repetitions=repetitions,
    phase_mode=phase_mode,
    freq_target=freq_target,
    use_qft=use_qft,
    qft_inverse=qft_inverse,
)

print("num_qubits:", qc_v6.num_qubits)
print("depth:", qc_v6.depth())
print("gate counts:", qc_v6.count_ops())

fig = qc_v6.draw("mpl")
fig.savefig(V6_OUTPUT_DIR / "v6_circuit.png", dpi=200, bbox_inches="tight")
plt.close(fig)

sv = Statevector(qc_v6)
try:
    display(sv.draw(output="latex", max_size=2**qc_v6.num_qubits, prefix="|\\psi\\rangle = "))
except Exception as exc:
    print(f"Skipping LaTeX statevector rendering: {exc}")


In [ ]:
# =========================================================
# ANALYSIS 2 — Statevector analysis
# =========================================================

sv_v6 = Statevector.from_instruction(qc_v6)
probs_v6 = index_probabilities_from_statevector(sv_v6, meta_v6)

valid_indices_v6 = [i for i, v in enumerate(meta_v6["valids"]) if v == 1]
P_valid_before = len(valid_indices_v6) / meta_v6["W"]
P_valid_after = float(sum(probs_v6[i] for i in valid_indices_v6))
P_invalid_index_after = float(sum(probs_v6[i] for i in range(meta_v6["W"], len(probs_v6))))

print(f"N={meta_v6['N']}, M={meta_v6['M']}, W={meta_v6['W']}, IDX={meta_v6['IDX']}")
print(f"phase_mode={meta_v6['phase_mode']}, qft_inverse={meta_v6['qft_inverse']}")
print(f"theta={meta_v6['theta']:.6g}, exp_base={meta_v6['exp_base']:.6g}, repetitions={meta_v6['repetitions']}")
print()
print("index | start coord | window string | C(i) | amplitude | probability")
print("------|-------------|---------------|------|-----------|------------")

for i, start in enumerate(meta_v6["starts"]):
    window = window_string_classical(meta_v6["grid_bits"], start, meta_v6["N"], meta_v6["M"])
    amp = sv_v6.data[i] if i < len(sv_v6.data) else 0.0
    print(f"{i:5d} | {str(start):11s} | {window:13s} | {meta_v6['costs'][i]:4d} | {amp.real:+.6f}{amp.imag:+.6f}j | {probs_v6[i]:10.6f}")

print()
print(f"K={len(valid_indices_v6)}, K/W={P_valid_before:.6f}")
print(f"P_valid_before = {P_valid_before:.6f}")
print(f"P_valid_after  = {P_valid_after:.6f}")
if P_invalid_index_after > 1e-12:
    print(f"P_invalid_index_after = {P_invalid_index_after:.6f}")
if P_valid_after < P_valid_before:
    print("Warning: P_valid_after < P_valid_before; retune theta, repetitions, or phase_mode.")

analysis_v6 = {
    "statevector": sv_v6,
    "probs": probs_v6,
    "valid_indices": valid_indices_v6,
    "P_valid_before": P_valid_before,
    "P_valid_after": P_valid_after,
    "P_uniform": P_valid_before,
    "P_invalid_index_after": P_invalid_index_after,
}


In [ ]:
# =========================================================
# ANALYSIS 3 — Phase profile visualization
# =========================================================

x = np.arange(meta_v6["W"])
colors = [COLOR_VALID if v == 1 else COLOR_INVALID for v in meta_v6["valids"]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(x, meta_v6["phi_profile"], color=colors)
axes[0].set_title("Phase profile phi_i")
axes[0].set_xlabel("index i")
axes[0].set_ylabel("phi_i")

axes[1].bar(x, meta_v6["costs"], color=colors)
axes[1].set_title("Cost profile C(i)")
axes[1].set_xlabel("index i")
axes[1].set_ylabel("C(i)")

plt.tight_layout()
fig.savefig(V6_OUTPUT_DIR / "v6_phase_profile.png", dpi=200, bbox_inches="tight")
fig.savefig(V6_OUTPUT_DIR / "v6_phase_profile.pdf", bbox_inches="tight")
plt.close(fig)


In [ ]:
# =========================================================
# ANALYSIS 4 — Amplitude distribution before vs after
# =========================================================

x = np.arange(meta_v6["W"])
before_probs = np.ones(meta_v6["W"]) / meta_v6["W"]
after_probs = analysis_v6["probs"][:meta_v6["W"]]
colors = [COLOR_VALID if v == 1 else COLOR_INVALID for v in meta_v6["valids"]]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - 0.18, before_probs, width=0.36, color=colors, alpha=0.30, label="before")
ax.bar(x + 0.18, after_probs, width=0.36, color=colors, alpha=0.90, label="after")
ax.axhline(1 / meta_v6["W"], color=COLOR_BASELINE, linestyle="--", linewidth=1.5, label="1/W")
ax.set_title("Amplitude distribution before vs after")
ax.set_xlabel("index i")
ax.set_ylabel(r"probability $|\langle i|\Psi\rangle|^2$")
ax.set_xticks(x)
ax.legend()

plt.tight_layout()
fig.savefig(V6_OUTPUT_DIR / "v6_amplitude_dist.png", dpi=200, bbox_inches="tight")
fig.savefig(V6_OUTPUT_DIR / "v6_amplitude_dist.pdf", bbox_inches="tight")
plt.close(fig)


In [ ]:
# =========================================================
# ANALYSIS 5 — Phase mode comparison
# =========================================================

modes = ["cost_linear", "cost_exponential", "delta_cost", "fourier_encoded"]
mode_scores = []
mode_metadata = {}

for mode in modes:
    qc_mode, meta_mode = build_v6_circuit(
        N=N, M=M, occupied_coords=occupied_coords,
        theta=theta, exp_base=exp_base, repetitions=repetitions,
        phase_mode=mode, freq_target=freq_target, use_qft=use_qft,
        qft_inverse=qft_inverse,
    )
    p_valid, _ = p_valid_from_circuit(qc_mode, meta_mode)
    mode_scores.append(p_valid)
    mode_metadata[mode] = meta_mode

P_uniform = analysis_v6["P_uniform"]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(modes, mode_scores, color=["#3498db", "#9b59b6", "#1abc9c", "#f39c12"], alpha=0.85)
ax.axhline(P_uniform, color=COLOR_BASELINE, linestyle="--", linewidth=1.5, label="P_uniform")
ax.set_ylim(0, max(1.0, max(mode_scores) * 1.15))
ax.set_title("Pure QFT phase mode comparison")
ax.set_xlabel("phase_mode")
ax.set_ylabel("P_valid")
ax.legend()
for bar, value in zip(bars, mode_scores):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.015, f"{value:.3f}", ha="center", va="bottom", fontsize=11)

plt.tight_layout()
fig.savefig(V6_OUTPUT_DIR / "v6_mode_comparison.png", dpi=200, bbox_inches="tight")
fig.savefig(V6_OUTPUT_DIR / "v6_mode_comparison.pdf", bbox_inches="tight")
plt.close(fig)


In [ ]:
# =========================================================
# ANALYSIS 6 — QFT readout vs no-QFT phase-only comparison
# =========================================================

rs = np.arange(1, 21)
scores_qft = []
scores_iqft = []
scores_no_qft = []

for r in rs:
    qc_qft, meta_qft = build_v6_circuit(
        N=N, M=M, occupied_coords=occupied_coords,
        theta=theta, exp_base=exp_base, repetitions=int(r),
        phase_mode=phase_mode, freq_target=freq_target, use_qft=True,
        qft_inverse=False,
    )
    p_qft, _ = p_valid_from_circuit(qc_qft, meta_qft)
    scores_qft.append(p_qft)

    qc_iqft, meta_iqft = build_v6_circuit(
        N=N, M=M, occupied_coords=occupied_coords,
        theta=theta, exp_base=exp_base, repetitions=int(r),
        phase_mode=phase_mode, freq_target=freq_target, use_qft=True,
        qft_inverse=True,
    )
    p_iqft, _ = p_valid_from_circuit(qc_iqft, meta_iqft)
    scores_iqft.append(p_iqft)

    qc_no_qft, meta_no_qft = build_v6_circuit(
        N=N, M=M, occupied_coords=occupied_coords,
        theta=theta, exp_base=exp_base, repetitions=int(r),
        phase_mode=phase_mode, freq_target=freq_target, use_qft=False,
    )
    p_no_qft, _ = p_valid_from_circuit(qc_no_qft, meta_no_qft)
    scores_no_qft.append(p_no_qft)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(rs, scores_qft, marker="o", label="QFT readout")
ax.plot(rs, scores_iqft, marker="s", label="iQFT readout")
ax.plot(rs, scores_no_qft, marker="^", label="phase only, no QFT")
ax.axhline(P_uniform, color=COLOR_BASELINE, linestyle="--", linewidth=1.5, label="P_uniform")
ax.set_title("Pure QFT readout: P_valid vs repetitions")
ax.set_xlabel("repetitions")
ax.set_ylabel("P_valid")
ax.set_xticks(rs)
ax.set_ylim(0, 1.05)
ax.legend()

plt.tight_layout()
fig.savefig(V6_OUTPUT_DIR / "v6_qft_comparison.png", dpi=200, bbox_inches="tight")
fig.savefig(V6_OUTPUT_DIR / "v6_qft_comparison.pdf", bbox_inches="tight")
plt.close(fig)


## Case-study analysis

This section keeps the V4/V5 benchmark family, but V6 is now a pure QFT readout model. The reduced model matches the compiled circuit: repeated diagonal phase accumulation followed by one final QFT or iQFT. There is no local mixer and no beta parameter.


In [ ]:
# =========================================================
# V6 case studies: same benchmark family as V4/V5
# =========================================================

V6_CASE_STUDIES = [
    {
        "name": "01_1d_tiny_single_gap",
        "description": "Minimal 1D case with one valid window.",
        "N": [6], "M": [2],
        "occupied_coords": [(0,), (3,), (4,)],
        "theta": np.pi / 2,
    },
    {
        "name": "02_1d_main_reference",
        "description": "Reference 1D instance used in V4/V5.",
        "N": [8], "M": [2],
        "occupied_coords": [(0,), (1,), (2,), (6,), (7,)],
        "theta": np.pi / 3,
    },
    {
        "name": "03_1d_two_free_regions",
        "description": "1D grid with two occupied blocks and a central valid plateau.",
        "N": [10], "M": [3],
        "occupied_coords": [(0,), (1,), (7,), (8,), (9,)],
        "theta": np.pi / 3,
    },
    {
        "name": "04_1d_clustered_medium",
        "description": "Medium 1D case with several clustered obstacles.",
        "N": [16], "M": [3],
        "occupied_coords": [(0,), (1,), (5,), (6,), (7,), (13,), (14,)],
        "theta": np.pi / 3,
    },
    {
        "name": "05_1d_long_clustered_blocks",
        "description": "Longer 1D case with separated occupied blocks.",
        "N": [32], "M": [4],
        "occupied_coords": [(0,), (1,), (2,), (9,), (10,), (11,), (18,), (19,), (28,), (29,), (30,), (31,)],
        "theta": np.pi / 4,
    },
    {
        "name": "06_2d_tiny_corner_block",
        "description": "Small 2D case with a single valid 2x2 region.",
        "N": [3, 3], "M": [2, 2],
        "occupied_coords": [(0, 0), (0, 2), (2, 2)],
        "theta": np.pi / 2,
    },
    {
        "name": "07_2d_small_diagonal_block",
        "description": "4x4 grid with diagonal obstacles and one clear lower-left solution.",
        "N": [4, 4], "M": [2, 2],
        "occupied_coords": [(1, 1), (2, 2), (0, 3)],
        "theta": np.pi / 2,
    },
    {
        "name": "08_2d_medium_clustered_obstacles",
        "description": "5x5 grid with two compact occupied clusters.",
        "N": [5, 5], "M": [2, 2],
        "occupied_coords": [(0, 0), (0, 1), (1, 0), (3, 3), (3, 4), (4, 3)],
        "theta": np.pi / 3,
    },
    {
        "name": "09_2d_rectangular_window",
        "description": "6x6 grid with a non-square 3x2 window.",
        "N": [6, 6], "M": [3, 2],
        "occupied_coords": [(0, 0), (0, 1), (1, 0), (4, 4), (4, 5), (5, 4), (2, 3)],
        "theta": np.pi / 3,
    },
    {
        "name": "10_3d_small_clustered_obstacles",
        "description": "Small 3D grid with two compact occupied clusters.",
        "N": [4, 4, 3], "M": [2, 2, 2],
        "occupied_coords": [(0, 0, 0), (0, 0, 1), (1, 0, 0), (3, 3, 2), (3, 2, 2), (2, 3, 2)],
        "theta": np.pi / 4,
    },
]

# Default V6 strategy for the full benchmark. The earlier phase-mode
# comparison cell includes the V5-inspired delta_cost mode for the main case.
V6_DEFAULT_PHASE_MODE = "cost_linear"
V6_DEFAULT_QFT_SANDWICH = True
V6_DEFAULT_USE_QFT = True
V6_DEFAULT_QFT_INVERSE = False
V6_DEFAULT_EXP_BASE = 2.0
V6_DEFAULT_SHIFT = 1
V6_DEFAULT_BOUNDARY_MODE = "zero"


## Shared V6 analysis utilities

These utilities mirror the V4/V5 reduced-model approach, but the layer matrix includes the V6 QFT transform. They do not produce output until the final case-study cell is executed.


In [ ]:
# =========================================================
# Shared reduced-model and final-circuit utilities for pure V6
# =========================================================

def v6_slug(text):
    return re.sub(r"[^a-zA-Z0-9_]+", "_", text).strip("_").lower()


def v6_savefig(fig, stem, output_dir=None):
    out_dir = Path(output_dir) if output_dir is not None else V6_OUTPUT_DIR
    out_dir.mkdir(exist_ok=True)
    fig.savefig(out_dir / f"{stem}.pdf", bbox_inches="tight")
    fig.savefig(out_dir / f"{stem}.png", dpi=200, bbox_inches="tight")
    plt.close(fig)


def v6_write_dict_rows_csv(path, rows):
    if not rows:
        return
    fieldnames = []
    for row in rows:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)
    with open(path, "w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def v6_case_context(case, phase_mode=None, use_qft=None):
    N_case = list(case["N"])
    M_case = list(case["M"])
    occupied_case = list(case["occupied_coords"])
    validate_problem(N_case, M_case)
    starts_case = valid_starts_nd(N_case, M_case)
    grid_bits_case = build_grid_bits(N_case, occupied_case)
    costs_case = [
        compute_window_cost_classical(grid_bits_case, s, N_case, M_case)
        for s in starts_case
    ]
    valid_indices_case = [i for i, c in enumerate(costs_case) if c == 0]
    if not valid_indices_case:
        raise ValueError(f"Case {case['name']} has no valid windows; change occupied_coords.")
    W_case = len(starts_case)
    IDX_case = int(np.ceil(np.log2(W_case))) if W_case > 1 else 1
    shift_case = int(case.get("shift", V6_DEFAULT_SHIFT))
    boundary_mode_case = case.get("boundary_mode", V6_DEFAULT_BOUNDARY_MODE)
    return {
        "name": case["name"],
        "description": case.get("description", ""),
        "N": N_case,
        "M": M_case,
        "occupied_coords": occupied_case,
        "starts": starts_case,
        "grid_bits": grid_bits_case,
        "costs": costs_case,
        "delta_costs": compute_delta_costs_classical(
            costs_case, shift=shift_case, boundary_mode=boundary_mode_case,
        ),
        "valid_indices": valid_indices_case,
        "W": W_case,
        "IDX": IDX_case,
        "P_uniform": len(valid_indices_case) / W_case,
        "theta_default": float(case.get("theta", globals().get("theta", np.pi / 3))),
        "exp_base": float(case.get("exp_base", V6_DEFAULT_EXP_BASE)),
        "phase_mode": phase_mode or case.get("phase_mode", V6_DEFAULT_PHASE_MODE),
        "freq_target": case.get("freq_target", None),
        "shift": shift_case,
        "boundary_mode": boundary_mode_case,
        "use_qft": V6_DEFAULT_USE_QFT if use_qft is None else bool(use_qft),
        "qft_inverse": bool(case.get("qft_inverse", V6_DEFAULT_QFT_INVERSE)),
    }


def v6_qft_matrix(W, inverse=False):
    if W <= 1:
        return np.eye(W, dtype=complex)
    j, k = np.meshgrid(np.arange(W), np.arange(W), indexing="ij")
    F = np.exp(2j * np.pi * j * k / W) / np.sqrt(W)
    return F.conj().T if inverse else F


def v6_phase_vector(ctx, theta_value):
    phases, selected_freq = compute_phase_profile(
        ctx["costs"],
        theta=float(theta_value),
        exp_base=ctx.get("exp_base", V6_DEFAULT_EXP_BASE),
        phase_mode=ctx.get("phase_mode", V6_DEFAULT_PHASE_MODE),
        freq_target=ctx.get("freq_target", None),
        shift=ctx.get("shift", V6_DEFAULT_SHIFT),
        boundary_mode=ctx.get("boundary_mode", V6_DEFAULT_BOUNDARY_MODE),
    )
    ctx["selected_freq"] = selected_freq
    return np.asarray(phases, dtype=float)


def v6_layer_matrix(ctx, theta_value):
    return np.diag(np.exp(1j * v6_phase_vector(ctx, theta_value)))


def v6_initial_state(ctx):
    return np.ones(ctx["W"], dtype=complex) / np.sqrt(ctx["W"])


def v6_final_readout_matrix(ctx):
    if ctx.get("use_qft", True):
        return v6_qft_matrix(ctx["W"], inverse=ctx.get("qft_inverse", False))
    return np.eye(ctx["W"], dtype=complex)


def v6_probabilities(ctx, theta_value, reps_value):
    U = v6_layer_matrix(ctx, theta_value)
    psi = v6_initial_state(ctx)
    for _ in range(int(reps_value)):
        psi = U @ psi
    psi = v6_final_readout_matrix(ctx) @ psi
    return np.abs(psi) ** 2


def v6_p_valid(ctx, probs):
    return float(np.sum(np.asarray(probs)[ctx["valid_indices"]]))


def v6_final_circuit_simulation(ctx, theta_opt, r_opt, force_statevector=False):
    reduced_probs = v6_probabilities(ctx, theta_opt, r_opt)
    reduced_p_valid = v6_p_valid(ctx, reduced_probs)
    simulate_qiskit = bool(globals().get("V6_SIMULATE_QISKIT_CIRCUIT", False))

    if not simulate_qiskit and not force_statevector:
        return {
            "circuit": None,
            "metadata": None,
            "probs": reduced_probs,
            "p_valid": reduced_p_valid,
            "p_valid_reduced": reduced_p_valid,
            "qiskit_statevector_used": False,
            "qiskit_circuit_built": False,
            "circuit_depth": None,
            "circuit_gates": {},
            "num_qubits": ctx["IDX"],
        }

    qc, metadata = build_v6_circuit(
        ctx["N"], ctx["M"], ctx["occupied_coords"],
        theta_opt, ctx.get("exp_base", V6_DEFAULT_EXP_BASE), r_opt,
        phase_mode=ctx.get("phase_mode", V6_DEFAULT_PHASE_MODE),
        freq_target=ctx.get("freq_target", None),
        use_qft=ctx.get("use_qft", True),
        qft_inverse=ctx.get("qft_inverse", False),
        shift=ctx.get("shift", V6_DEFAULT_SHIFT),
        boundary_mode=ctx.get("boundary_mode", V6_DEFAULT_BOUNDARY_MODE),
        add_barriers=False,
    )
    circuit_depth = qc.depth()
    circuit_gates = dict(qc.count_ops())

    qubit_limit = globals().get("V6_QISKIT_STATEVECTOR_QUBIT_LIMIT", 18)
    if qc.num_qubits > qubit_limit and not force_statevector:
        return {
            "circuit": qc,
            "metadata": metadata,
            "probs": reduced_probs,
            "p_valid": reduced_p_valid,
            "p_valid_reduced": reduced_p_valid,
            "qiskit_statevector_used": False,
            "qiskit_circuit_built": True,
            "circuit_depth": circuit_depth,
            "circuit_gates": circuit_gates,
            "num_qubits": qc.num_qubits,
        }

    sv = Statevector(qc)
    idx_probs = index_probabilities_from_statevector(sv, metadata)[: ctx["W"]]
    p_valid_final = v6_p_valid(ctx, idx_probs)
    tolerance = globals().get("V6_MODEL_TOL", 1e-6)
    if abs(p_valid_final - reduced_p_valid) > tolerance:
        print(
            f"[V6 warning] {ctx['name']}: reduced model P_valid "
            f"{reduced_p_valid:.6f} differs from Qiskit Statevector {p_valid_final:.6f}."
        )
    return {
        "circuit": qc,
        "metadata": metadata,
        "probs": idx_probs,
        "p_valid": p_valid_final,
        "p_valid_reduced": reduced_p_valid,
        "qiskit_statevector_used": True,
        "qiskit_circuit_built": True,
        "circuit_depth": circuit_depth,
        "circuit_gates": circuit_gates,
        "num_qubits": qc.num_qubits,
    }

print("Pure V6 QFT shared analysis utilities loaded.")


## Final V6 QFT optimiser

This optimiser is adapted to the pure-QFT V6 circuit. It searches only `theta` and repetitions `r`; there is no `beta` because there is no mixer.


In [ ]:
# =========================================================
# Final theta/r optimiser for pure V6 case studies
# =========================================================

V6_HYBRID_THETA_POINTS = 31
V6_HYBRID_R_MAX = 100
V6_HYBRID_TOP_K = 12
V6_HYBRID_REFINE_ROUNDS = 3
V6_HYBRID_REFINE_GRID_POINTS = 7
V6_HARDWARE_P_TOL = 0.03
V6_HARDWARE_LAMBDA = 0.002
V6_SIMULATE_QISKIT_CIRCUIT = False
V6_QISKIT_STATEVECTOR_QUBIT_LIMIT = 18
V6_MODEL_TOL = 1e-6


def v6_candidate_hardware_score(candidate, hardware_lambda=V6_HARDWARE_LAMBDA):
    return float(candidate["p_valid"] - hardware_lambda * int(candidate["r"]))


def v6_evaluate_theta_with_best_r(ctx, theta, r_max):
    theta = float(np.clip(theta, 0.0, np.pi))
    U = v6_layer_matrix(ctx, theta)
    psi = v6_initial_state(ctx)
    readout = v6_final_readout_matrix(ctx)
    p_curve = np.zeros(int(r_max), dtype=float)
    best_r = 1
    best_p = -np.inf

    for idx in range(int(r_max)):
        psi = U @ psi
        probs = np.abs(readout @ psi) ** 2
        p_value = v6_p_valid(ctx, probs)
        p_curve[idx] = p_value
        if p_value > best_p:
            best_p = float(p_value)
            best_r = idx + 1

    return {
        "theta": theta,
        "r": int(best_r),
        "best_r": int(best_r),
        "p_valid": float(best_p),
        "best_P_valid": float(best_p),
        "p_curve": p_curve,
    }


def v6_coarse_global_search(ctx, theta_points, r_max, top_k):
    theta_values = np.linspace(0.0, np.pi, int(theta_points))
    heatmap = np.zeros((len(theta_values), int(r_max)), dtype=float)
    candidates = []

    for i, theta_value in enumerate(theta_values):
        candidate = v6_evaluate_theta_with_best_r(ctx, theta_value, r_max)
        heatmap[i, :] = candidate["p_curve"]
        candidates.append(candidate)

    candidates.sort(key=lambda row: (row["p_valid"], -row["r"]), reverse=True)
    return {
        "candidates": candidates[: int(top_k)],
        "all_candidates": candidates,
        "heatmap": heatmap,
        "theta_values": theta_values,
        "r_values": np.arange(1, int(r_max) + 1),
    }


def v6_deduplicate_candidates(candidates, ndigits=12):
    unique = {}
    for candidate in candidates:
        key = (round(float(candidate["theta"]), ndigits), int(candidate["r"]))
        if key not in unique or candidate["p_valid"] > unique[key]["p_valid"]:
            unique[key] = candidate
    return list(unique.values())


def v6_local_refinement(ctx, candidates, r_max, rounds, initial_radius_theta, grid_points):
    expanded = []
    seeds = sorted(candidates, key=lambda row: row["p_valid"], reverse=True)

    for seed in seeds:
        current = seed
        radius_theta = float(initial_radius_theta)
        expanded.append(seed)

        for _round in range(int(rounds)):
            theta_offsets = np.linspace(-radius_theta, radius_theta, int(grid_points))
            local_candidates = []

            for dtheta in theta_offsets:
                theta_test = float(np.clip(current["theta"] + dtheta, 0.0, np.pi))
                local_candidates.append(v6_evaluate_theta_with_best_r(ctx, theta_test, r_max))

            local_candidates = v6_deduplicate_candidates(local_candidates)
            local_candidates.sort(
                key=lambda row: (row["p_valid"], v6_candidate_hardware_score(row), -row["r"]),
                reverse=True,
            )
            expanded.extend(local_candidates)
            best_local = local_candidates[0]

            current_score = v6_candidate_hardware_score(current)
            best_score = v6_candidate_hardware_score(best_local)
            improves_probability = best_local["p_valid"] > current["p_valid"] + 1e-12
            keeps_probability_and_scores_better = (
                abs(best_local["p_valid"] - current["p_valid"]) <= 1e-12
                and best_score > current_score
            )
            if improves_probability or keeps_probability_and_scores_better:
                current = best_local

            radius_theta *= 0.5

    expanded = v6_deduplicate_candidates(expanded)
    expanded.sort(
        key=lambda row: (row["p_valid"], v6_candidate_hardware_score(row), -row["r"]),
        reverse=True,
    )
    return expanded


def v6_select_hardware_aware_candidate(candidates, p_tol, hardware_lambda):
    if not candidates:
        raise ValueError("No candidates available for hardware-aware selection.")
    p_max = max(float(candidate["p_valid"]) for candidate in candidates)
    eligible = [
        dict(candidate)
        for candidate in candidates
        if float(candidate["p_valid"]) >= p_max - float(p_tol)
    ]
    for candidate in eligible:
        candidate["score"] = v6_candidate_hardware_score(candidate, hardware_lambda)
        candidate["P_max"] = p_max
    eligible.sort(key=lambda row: (row["score"], row["p_valid"], -row["r"]), reverse=True)
    return eligible[0]


def v6_best_first_hybrid_optimizer(ctx, theta_points=V6_HYBRID_THETA_POINTS,
                                   r_max=V6_HYBRID_R_MAX,
                                   top_k=V6_HYBRID_TOP_K,
                                   refine_rounds=V6_HYBRID_REFINE_ROUNDS,
                                   refine_grid_points=V6_HYBRID_REFINE_GRID_POINTS,
                                   p_tol=V6_HARDWARE_P_TOL,
                                   hardware_lambda=V6_HARDWARE_LAMBDA):
    coarse = v6_coarse_global_search(ctx, theta_points, r_max, top_k)
    initial_radius_theta = np.pi / max(1, int(theta_points) - 1)
    refined = v6_local_refinement(
        ctx,
        coarse["candidates"],
        r_max=r_max,
        rounds=refine_rounds,
        initial_radius_theta=initial_radius_theta,
        grid_points=refine_grid_points,
    )
    all_candidates = v6_deduplicate_candidates(coarse["all_candidates"] + refined)
    selected = v6_select_hardware_aware_candidate(all_candidates, p_tol, hardware_lambda)
    selected = v6_evaluate_theta_with_best_r(ctx, selected["theta"], r_max)
    selected["score"] = v6_candidate_hardware_score(selected, hardware_lambda)
    selected["P_max"] = max(candidate["p_valid"] for candidate in all_candidates)
    return {
        "selected": selected,
        "coarse": coarse,
        "refined_candidates": refined,
        "all_candidates": all_candidates,
    }


def v6_spectrum_data(ctx, theta_opt):
    U = v6_layer_matrix(ctx, theta_opt)
    eigvals = np.linalg.eigvals(U)
    phases = np.sort(np.angle(eigvals))
    wrapped = np.r_[phases, phases[0] + 2 * np.pi]
    gaps = np.diff(wrapped)
    return {
        "eigvals": eigvals,
        "phases": phases,
        "gaps": gaps,
    }


def v6_run_best_first_hybrid_case(ctx):
    optimizer_result = v6_best_first_hybrid_optimizer(ctx)
    selected = optimizer_result["selected"]
    theta_opt = selected["theta"]
    r_opt = selected["r"]
    final = v6_final_circuit_simulation(ctx, theta_opt, r_opt)
    spectrum = v6_spectrum_data(ctx, theta_opt)

    return {
        "case": ctx["name"],
        "ctx": ctx,
        "theta_opt": theta_opt,
        "r_opt": r_opt,
        "r_star": r_opt,
        "P_valid": final["p_valid"],
        "P_valid_reduced": final["p_valid_reduced"],
        "P_uniform": ctx["P_uniform"],
        "heatmap": optimizer_result["coarse"]["heatmap"],
        "theta_values": optimizer_result["coarse"]["theta_values"],
        "r_values": optimizer_result["coarse"]["r_values"],
        "oscillation_curve": selected["p_curve"],
        "spectrum": spectrum,
        "final_probs": final["probs"],
        "circuit_depth": final["circuit_depth"],
        "circuit_gates": final["circuit_gates"],
        "num_qubits": final["num_qubits"],
        "qiskit_statevector_used": final["qiskit_statevector_used"],
        "qiskit_circuit_built": final["qiskit_circuit_built"],
        "hardware_score": selected["score"],
        "P_max_reduced_search": selected["P_max"],
        "optimizer_result": optimizer_result,
    }


## Interpretation notes

The high probabilities reported by the final case-study section are ideal reduced-model optimisation results. They are useful for comparing whether a pure QFT readout can concentrate amplitude after tuning, but they should not be read as a blind end-to-end quantum search advantage.

The optimiser uses the known valid-window set to score candidates through `P_valid` while sweeping `theta` and repetitions `r`. This is the same benchmarking style used in V4/V5, but it means the parameter search is supervised by the answer. A deployment-style algorithm would need a parameter-selection rule that does not use the valid labels, or it should report the classical tuning cost separately.

V6 is now QFT-only in its quantum transformation after phase encoding: repeated diagonal phase accumulation followed by a final QFT/iQFT readout. There is no local mixer and no Grover diffusion operator.


## Final V6 case-study analysis

The following cell runs the final benchmark. It is safe to leave `V6_SIMULATE_QISKIT_CIRCUIT = False` for routine work; the reduced model is the intended fast optimisation path, and the full circuit can be rebuilt for selected small cases.


In [ ]:
# =========================================================
# Final pure-V6 case-study execution and plots
# =========================================================

def v6_draw_oscillation_plot(result, ax):
    reps = np.arange(1, len(result["oscillation_curve"]) + 1)
    ax.plot(reps, result["oscillation_curve"], marker="o", linewidth=2,
            color=COLOR_VALID, label="P_valid")
    ax.axvline(result["r_opt"], color="black", linestyle=":", linewidth=1.5,
               label=f"r_opt={result['r_opt']}")
    ax.axhline(result["P_uniform"], color=COLOR_BASELINE, linestyle="--",
               linewidth=1.5, label="uniform K/W")
    ax.set_xlabel("repetitions r")
    ax.set_ylabel("P_valid")
    ax.set_title(
        f"Pure V6 QFT oscillation - {result['case']}\n"
        f"phase={result['ctx']['phase_mode']}, theta={result['theta_opt']/np.pi:.3f}pi, "
        f"r={result['r_opt']}"
    )
    ax.legend(fontsize=9)


def v6_draw_heatmap_plot(result, ax):
    theta_values = result["theta_values"]
    r_values = result["r_values"]
    im = ax.imshow(
        result["heatmap"],
        origin="lower",
        aspect="auto",
        cmap="viridis",
        extent=[
            r_values[0],
            r_values[-1],
            theta_values[0] / np.pi,
            theta_values[-1] / np.pi,
        ],
    )
    ax.scatter([result["r_opt"]], [result["theta_opt"] / np.pi],
               marker="*", s=180, color="white", edgecolor="black",
               linewidth=0.8, label="selected")
    ax.annotate(
        f"r={result['r_opt']}",
        xy=(result["r_opt"], result["theta_opt"] / np.pi),
        xytext=(6, 6),
        textcoords="offset points",
        color="white",
        fontsize=9,
        weight="bold",
    )
    ax.set_xlabel("repetitions r")
    ax.set_ylabel("theta / pi")
    ax.set_title(f"Pure V6 QFT heatmap - {result['case']}")
    ax.legend(loc="lower right", fontsize=9)
    return im


def v6_draw_spectrum_plot(result, ax):
    eigvals = result["spectrum"]["eigvals"]
    phases = np.angle(eigvals) / np.pi
    unit = np.exp(1j * np.linspace(0, 2 * np.pi, 400))
    ax.plot(unit.real, unit.imag, color="0.75", linewidth=1.2)
    sc = ax.scatter(eigvals.real, eigvals.imag, c=phases, cmap="twilight",
                    s=55, edgecolor="black", linewidth=0.3)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Re(lambda)")
    ax.set_ylabel("Im(lambda)")
    ax.set_title(f"Pure V6 phase spectrum - {result['case']}")
    return sc


def v6_draw_distribution_plot(result, ax):
    ctx = result["ctx"]
    probs = np.asarray(result["final_probs"], dtype=float)
    bar_colors = [
        COLOR_VALID if i in ctx["valid_indices"] else COLOR_INVALID
        for i in range(ctx["W"])
    ]
    ax.bar(np.arange(ctx["W"]), probs[: ctx["W"]], color=bar_colors)
    ax.axhline(1 / ctx["W"], color=COLOR_BASELINE, linestyle="--",
               linewidth=1.5, label="uniform 1/W")
    ax.set_xlabel("window index")
    ax.set_ylabel("probability")
    ax.set_title(
        f"Pure V6 QFT distribution - {result['case']}\n"
        f"P={result['P_valid']:.4f}, theta={result['theta_opt']/np.pi:.3f}pi, "
        f"r={result['r_opt']}"
    )
    ax.legend(fontsize=9)


def v6_save_case_figures(result):
    slug = v6_slug(result["case"])

    fig, ax = plt.subplots(figsize=(7, 4.5))
    v6_draw_oscillation_plot(result, ax)
    fig.tight_layout()
    v6_savefig(fig, f"{slug}_oscillation")

    fig, ax = plt.subplots(figsize=(7, 4.8))
    im = v6_draw_heatmap_plot(result, ax)
    fig.colorbar(im, ax=ax, label="P_valid")
    fig.tight_layout()
    v6_savefig(fig, f"{slug}_heatmap")

    fig, ax = plt.subplots(figsize=(6, 5.2))
    sc = v6_draw_spectrum_plot(result, ax)
    fig.colorbar(sc, ax=ax, label="eigenphase / pi")
    fig.tight_layout()
    v6_savefig(fig, f"{slug}_spectrum")

    fig, ax = plt.subplots(figsize=(8, 4.5))
    v6_draw_distribution_plot(result, ax)
    fig.tight_layout()
    v6_savefig(fig, f"{slug}_optimal_distribution")

    fig, axes = plt.subplots(2, 2, figsize=(13, 9.5))
    v6_draw_oscillation_plot(result, axes[0, 0])
    im = v6_draw_heatmap_plot(result, axes[0, 1])
    fig.colorbar(im, ax=axes[0, 1], label="P_valid")
    sc = v6_draw_spectrum_plot(result, axes[1, 0])
    fig.colorbar(sc, ax=axes[1, 0], label="eigenphase / pi")
    v6_draw_distribution_plot(result, axes[1, 1])
    fig.suptitle(f"Pure V6 QFT summary - {result['case']}", fontsize=16)
    fig.tight_layout(rect=[0, 0.02, 1, 0.96])
    v6_savefig(fig, f"{slug}_summary")


def v6_save_cross_case_comparison(rows):
    names = [row["case"] for row in rows]
    x = np.arange(len(names))
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.bar(x - 0.2, [row["P_uniform"] for row in rows], width=0.4,
           color="0.7", label="Uniform K/W")
    ax.bar(x + 0.2, [row["P_valid"] for row in rows], width=0.4,
           color=COLOR_VALID, label="Pure V6 QFT optimal P_valid")
    for xi, row in zip(x, rows):
        ax.text(xi + 0.2, min(1.04, row["P_valid"] + 0.015),
                f"{row['P_valid']:.3f}", ha="center", va="bottom", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels([name.replace("_", "\n", 2) for name in names],
                       rotation=40, ha="right")
    ax.set_ylabel("P_valid")
    ax.set_ylim(0, 1.08)
    ax.set_title("Pure V6 QFT: Uniform K/W vs optimal P_valid")
    ax.legend()
    fig.tight_layout()
    v6_savefig(fig, "final_v6_qft_cross_case_comparison")


V6_FINAL_CASES = [v6_case_context(case) for case in V6_CASE_STUDIES]
v6_final_results = []
v6_final_rows = []
v6_curve_rows = []
v6_distribution_rows = []

for ctx in V6_FINAL_CASES:
    print(
        f"\n[Pure V6 QFT] {ctx['name']} | phase={ctx['phase_mode']} | "
        f"W={ctx['W']} | K={len(ctx['valid_indices'])} | uniform={ctx['P_uniform']:.4f}"
    )
    result = v6_run_best_first_hybrid_case(ctx)
    v6_final_results.append(result)
    v6_save_case_figures(result)

    row = {
        "version": "V6",
        "case": result["case"],
        "oracle": f"pure_qft_{ctx['phase_mode']}",
        "phase_mode": ctx["phase_mode"],
        "use_qft": ctx["use_qft"],
        "num_qubits": result["num_qubits"],
        "W": ctx["W"],
        "K": len(ctx["valid_indices"]),
        "P_uniform": result["P_uniform"],
        "theta_opt": result["theta_opt"],
        "theta_over_pi": result["theta_opt"] / np.pi,
        "r_opt": result["r_opt"],
        "r_star": result["r_opt"],
        "P_valid": result["P_valid"],
        "P_valid_reduced": result["P_valid_reduced"],
        "circuit_depth": result["circuit_depth"],
        "qiskit_statevector_used": result["qiskit_statevector_used"],
        "qiskit_circuit_built": result["qiskit_circuit_built"],
    }
    v6_final_rows.append(row)

    selected_curve = result["optimizer_result"]["selected"]["p_curve"]
    for rep_idx, p_value in enumerate(selected_curve, start=1):
        v6_curve_rows.append({
            "version": "V6",
            "case": result["case"],
            "phase_mode": ctx["phase_mode"],
            "theta_opt": result["theta_opt"],
            "theta_over_pi": result["theta_opt"] / np.pi,
            "r": rep_idx,
            "P_valid": float(p_value),
            "is_selected_r": int(rep_idx == result["r_opt"]),
        })

    final_probs = np.asarray(result["final_probs"], dtype=float)
    for window_idx, start in enumerate(ctx["starts"]):
        v6_distribution_rows.append({
            "version": "V6",
            "case": result["case"],
            "phase_mode": ctx["phase_mode"],
            "window_index": window_idx,
            "start": str(tuple(start)),
            "valid": window_idx in ctx["valid_indices"],
            "cost": ctx["costs"][window_idx],
            "delta_cost": ctx["delta_costs"][window_idx],
            "probability": float(final_probs[window_idx]),
        })

    print(f"  W:             {row['W']}")
    print(f"  K:             {row['K']}")
    print(f"  Uniform K/W:   {row['P_uniform']:.6f}")
    print(f"  theta_opt/pi:  {row['theta_over_pi']:.6f}")
    print(f"  r_opt:         {row['r_opt']}")
    print(f"  P_valid:       {row['P_valid']:.6f}")

    v6_write_dict_rows_csv(V6_OUTPUT_DIR / "v6_final_hybrid_results.csv", v6_final_rows)
    v6_write_dict_rows_csv(V6_OUTPUT_DIR / "v6_selected_repetition_curves.csv", v6_curve_rows)
    v6_write_dict_rows_csv(V6_OUTPUT_DIR / "v6_final_window_distribution.csv", v6_distribution_rows)

if v6_final_rows:
    print("\nPure V6 QFT summary")
    print("case | phase | W | K | Uniform K/W | theta/pi | r_opt | P_valid")
    print("-----|-------|---|---|-------------|----------|-------|--------")
    for row in v6_final_rows:
        print(
            f"{row['case']} | {row['phase_mode']} | {row['W']} | {row['K']} | "
            f"{row['P_uniform']:.4f} | {row['theta_over_pi']:.4f} | "
            f"{row['r_opt']} | {row['P_valid']:.4f}"
        )
    v6_save_cross_case_comparison(v6_final_rows)
    v6_write_dict_rows_csv(V6_OUTPUT_DIR / "v6_final_hybrid_results.csv", v6_final_rows)
    v6_write_dict_rows_csv(V6_OUTPUT_DIR / "v6_selected_repetition_curves.csv", v6_curve_rows)
    v6_write_dict_rows_csv(V6_OUTPUT_DIR / "v6_final_window_distribution.csv", v6_distribution_rows)
    print(f"CSV database files saved in: {V6_OUTPUT_DIR}")
    print(f"Final pure V6 figures saved in: {V6_OUTPUT_DIR}")
else:
    print("No pure V6 case studies completed.")
